# 02 - Trim/Filter Messages, Memory and External Memory

How to keep a message list from growing forever, and how a checkpointer gives a
graph short-term "memory" of a conversation thread.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## Trim Messages

`trim_messages` cuts a message list down to a token budget, keeping the most
recent messages by default.

In [ ]:
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, I'm Sri."),
    AIMessage(content="Hello Sri!"),
    HumanMessage(content="What's 2+2?"),
    AIMessage(content="4"),
    HumanMessage(content="What is LangGraph?"),
]

trimmed = trim_messages(messages, max_tokens=40, strategy="last", token_counter=llm, include_system=True)
for m in trimmed:
    print(type(m).__name__, ":", m.content)

## Filter Messages

Sometimes you just want to drop a category of message (e.g. system messages)
before sending to the model.

In [ ]:
filtered = [m for m in messages if not isinstance(m, SystemMessage)]
for m in filtered:
    print(type(m).__name__, ":", m.content)

## Memory and External Memory

A checkpointer persists graph state between `invoke` calls that share a
`thread_id` - that's the graph's short-term memory. `MemorySaver` keeps it in
RAM; swap in `SqliteSaver`/`PostgresSaver` for real external/persistent storage.

In [ ]:
def chat_node(state):
    trimmed = trim_messages(state["messages"], max_tokens=200, strategy="last", token_counter=llm, include_system=True)
    response = llm.invoke(trimmed)
    return {"messages": [response]}

memory = MemorySaver()

builder = StateGraph(MessagesState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)
chat_app = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "demo-thread-1"}}

chat_app.invoke({"messages": [HumanMessage(content="My name is Sri.")]}, config)
chat_app.invoke({"messages": [HumanMessage(content="What's my name?")]}, config)["messages"][-1].content